# Portfolio ESG Risk Scoring & Benchmarking Framework
### Asset Management & Corporate Sustainability Analytics

---

## 1. Project Executive Summary
In contemporary quantitative finance and institutional asset management, evaluating **Environmental, Social, and Governance (ESG)** risk exposure has evolved from a niche compliance checklist into a fundamental requirement for active risk management. Poor corporate oversight regarding carbon footprints, systemic supply chain human rights vulnerabilities, or internal governance collapses can trigger sharp capital depreciation and catastrophic reputational damage.

This project introduces a comprehensive, data-driven **ESG Risk Assessment Framework** designed to audit, score, and evaluate a synthetic multi-asset portfolio comprising **30 companies distributed across 5 core industrial sectors**:
* Technology
* Energy
* Financials
* Healthcare
* Manufacturing

By mathematically standardizing raw non-financial operational metrics into uniform risk scores ($0 - 100$), this framework generates an institutional-grade **ESG Scorecard**, computes dynamic **Sector Benchmarks**, flags high-exposure corporate **Laggards**, and maps portfolio concentrations via descriptive visual heat maps.

## 2. Quantitative Architecture & Weighting Methodology
To eliminate tracking biases across highly disparate raw dimensions (e.g., comparing metric tons of greenhouse gases directly to internal board diversity percentage ceilings), the framework employs an explicit **Min-Max Score Normalization Index**. 

Every calculated metric is mapped to a strict linear scale where **100 represents peak ESG compliance (Lowest Risk)** and **0 represents severe operational exposure (Highest Risk)**.

### Dynamic Weighting Formula
The Composite ESG Rating Matrix is determined by aggregating the independent operational pillars according to institutional asset allocation weights:

$$\text{Total ESG Score} = (E_{\text{Score}} \times 0.40) + (S_{\text{Score}} \times 0.30) + (G_{\text{Score}} \times 0.30)$$

---

### Pillar Breakdown Matrix

| Pillar | Underlying Raw Operational Metric Measured | Scaling Mechanism | Pillar Weight |
| :--- | :--- | :--- | :--- |
| **Environmental (E)** | **Carbon Intensity** <br>*(Metric tons $CO_2e$ per \$M revenue)* | **Inverse Scaling** <br>(Higher raw emissions lower the score toward $0$) | **40%** |
| **Social (S)** | **Board Diversity %** <br>**Supply Chain Risk Profile (1-100)** | **Balanced Index** <br>(High diversity increases score; high supply chain risk penalizes it) | **30%** |
| **Governance (G)** | **Structure Score (1-100)** <br>**Severe Controversy Incidents** | **Deductive Penalty Engine** <br>(Base corporate governance score minus linear penalties per flag) | **30%** |

### Risk Categorization Thresholds
Assets are continuously classified into a dynamic portfolio risk index based on their final weighted score outputs:
* $\text{Score} \ge 75 \implies$ **Leader (Low Risk Profile)**
* $50 \le \text{Score} < 75 \implies$ **Average (Medium Risk Profile)**
* $\text{Score} < 50 \implies$ **Laggard (High Capital Exposure)**

# 1: Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Set plotting style for clean portfolio reporting
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 3. Data Ingestion & Engineering Pipeline Initialization
The cell below constructs the underlying algorithmic engine. It injects specific statistical sector-realities (such as assigning structural carbon premiums to the *Energy* sector and low emissions to the *Technology* space) to ensure the synthetic portfolio accurately mirrors live institutional asset behaviors.

# Cell 2: Synthetic ESG Data Engine

In [ ]:
np.random.seed(42)

sectors = ['Technology', 'Energy', 'Financials', 'Healthcare', 'Manufacturing']
companies = [f"AlphaCorp {i}" for i in range(1, 31)]

data = {
    'Company': companies,
    'Sector': np.random.choice(sectors, 30),
    # Environmental: Carbon Intensity (Metric tons CO2e per $M revenue)
    'Carbon_Intensity': np.random.uniform(10, 500, 30), 
    # Social: Board Diversity (% of independent/diverse members)
    'Board_Diversity_Pct': np.random.uniform(15, 60, 30),
    # Social: Supply Chain Risk Score (1-100, lower is better risk)
    'Supply_Chain_Risk': np.random.uniform(20, 90, 30),
    # Governance: Governance Structure Score (1-100, higher is better)
    'Governance_Score': np.random.uniform(40, 95, 30),
    # Controversy Flags (0 to 3 severe incidents)
    'Controversy_Flags': np.random.choice([0, 1, 2, 3], 30, p=[0.6, 0.25, 0.1, 0.05])
}

df = pd.DataFrame(data)

# Adjust sector realities (e.g., Energy has higher carbon footprint)
df.loc[df['Sector'] == 'Energy', 'Carbon_Intensity'] *= 1.8
df.loc[df['Sector'] == 'Technology', 'Carbon_Intensity'] *= 0.3

print(f"Dataset initialized successfully. Shaped: {df.shape}")
df.head()

# Cell 3: ESG Risk Scoring Framework Architecture

In [ ]:
# Normalize metrics to a standardized 0-100 scale (Where 100 = Best ESG Performance)

# E-Score: Min-Max scaling reversed (High intensity = Low score)
df['E_Score'] = 100 * (1 - (df['Carbon_Intensity'] - df['Carbon_Intensity'].min()) / (df['Carbon_Intensity'].max() - df['Carbon_Intensity'].min()))

# S-Score: High diversity is good, high supply chain risk is bad (Equal weight)
s_diversity = 100 * (df['Board_Diversity_Pct'] - df['Board_Diversity_Pct'].min()) / (df['Board_Diversity_Pct'].max() - df['Board_Diversity_Pct'].min())
s_supply = 100 * (1 - (df['Supply_Chain_Risk'] - df['Supply_Chain_Risk'].min()) / (df['Supply_Chain_Risk'].max() - df['Supply_Chain_Risk'].min()))
df['S_Score'] = (s_diversity * 0.5) + (s_supply * 0.5)

# G-Score: High governance score is good. Subtract penalties for controversies.
g_base = 100 * (df['Governance_Score'] - df['Governance_Score'].min()) / (df['Governance_Score'].max() - df['Governance_Score'].min())
df['G_Score'] = g_base - (df['Controversy_Flags'] * 15)
df['G_Score'] = df['G_Score'].clip(lower=0) # Floor at 0

# Composite ESG Rating Matrix (Weights: E=40%, S=30%, G=30%)
df['Total_ESG_Score'] = (df['E_Score'] * 0.40) + (df['S_Score'] * 0.30) + (df['G_Score'] * 0.30)

# Risk Category Mapping
def classify_esg_risk(score):
    if score >= 75: return 'Leader (Low Risk)'
    elif score >= 50: return 'Average (Medium Risk)'
    else: return 'Laggard (High Risk)'

df['ESG_Risk_Class'] = df['Total_ESG_Score'].apply(classify_esg_risk)
df.to_csv('esg_portfolio_scorecard.csv', index=False)
print("ESG Scoring Engine execution complete. Scorecard compiled.")

# Cell 4: Portfolio Statistical Aggregations & Benchmarks

In [ ]:
sector_benchmarks = df.groupby('Sector')[['E_Score', 'S_Score', 'G_Score', 'Total_ESG_Score']].mean().round(2)
print("\n--- SECTOR BENCHMARKS ---")
print(sector_benchmarks)

risk_distribution = df['ESG_Risk_Class'].value_counts()
print("\n--- PORTFOLIO RISK DISTRIBUTION ---")
print(risk_distribution)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure output directory exists
os.makedirs("dashboards", exist_ok=True)

# 1. Load Generated Scorecard Data
try:
    df = pd.read_csv('esg_portfolio_scorecard.csv')
except FileNotFoundError:
    print("Error: Please run the Jupyter Notebook data generation code first.")
    exit()

# Set up global styling
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.titlesize'] = 16

# Cell 3: PORTFOLIO RISK HEATMAP & BENCHMARK DASHBOARD

In [ ]:
fig1, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left Subplot: Pivot Heatmap for Core ESG Matrix
pivot_df = df.groupby('Sector')[['E_Score', 'S_Score', 'G_Score']].mean()
sns.heatmap(pivot_df, annot=True, cmap="RdYlGn", fmt=".1f", linewidths=.5, cbar_kws={'label': 'Performance Score'}, ax=axes[0])
axes[0].set_title('Sector ESG Performance Matrix (Heat Map)', fontsize=13, fontweight='bold', pad=15)
axes[0].set_ylabel('Sector Investment Category')

# Right Subplot: Distribution of Portfolio Risk Classes
colors = ['#2ea44f', '#fca311', '#dc3545'] # Green, Orange, Red
risk_counts = df['ESG_Risk_Class'].value_counts()
axes[1].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', startangle=140, colors=colors, explode=(0.05, 0, 0))
axes[1].set_title('Portfolio Distribution by ESG Risk Profile', fontsize=13, fontweight='bold', pad=15)

plt.suptitle('ESG Portfolio Risk Profile & Pillar Benchmarks', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('dashboards/esg_portfolio_heatmap_dashboard.png', dpi=300)
plt.close()
print("Saved: dashboards/esg_portfolio_heatmap_dashboard.png")

# Cell 3: SECTOR BENCHMARKS & LAGGARD DETECTION

In [ ]:
fig2, axes = plt.subplots(2, 1, figsize=(14, 10))

# Upper Subplot: Total ESG Score Rankings across sectors
sns.boxplot(x='Sector', y='Total_ESG_Score', data=df, palette='Set2', ax=axes[0])
sns.stripplot(x='Sector', y='Total_ESG_Score', data=df, color='black', alpha=0.3, size=6, ax=axes[0])
axes[0].set_title('Total ESG Score Spread & Variances across Sectors', fontsize=13, fontweight='bold', pad=10)
axes[0].set_ylabel('Normalized Total ESG Rating (0-100)')

# Lower Subplot: Top 7 High-Risk Laggard Companies (Lowest ESG Scores)
laggards = df.sort_values(by='Total_ESG_Score').head(7)
sns.barplot(x='Total_ESG_Score', y='Company', hue='Sector', data=laggards, palette='rocket', dodge=False, ax=axes[1])
axes[1].axvline(50, color='red', linestyle='--', alpha=0.7, label='High Risk Threshold (<50)')
axes[1].set_title('Target List: Top 7 ESG Portfolio Laggards (High Risk Exposure)', fontsize=13, fontweight='bold', pad=10)
axes[1].set_xlabel('Total Risk Performance Score')
axes[1].legend(title='Sector Location')

plt.suptitle('Sector Benchmark Analytics & Risk Exposure Target Profiles', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('dashboards/esg_sector_benchmarks_laggards.png', dpi=300)
plt.close()
print("Saved: dashboards/esg_sector_benchmarks_laggards.png")